In [23]:
pip freeze

aiohappyeyeballs==2.7.1
aiohttp==3.14.3
aiosignal==1.4.0
annotated-types==0.8.0
anyio==4.15.0
argon2-cffi==25.1.0
argon2-cffi-bindings==26.1.0
arrow==1.4.0
asttokens==3.0.2
async-lru==2.3.0
async-timeout==4.0.3
attrs==26.1.0
babel==2.18.0
beautifulsoup4==4.15.0
bleach==6.4.0
certifi==2026.7.22
cffi==2.1.1
charset-normalizer==3.5.1
colorama==0.4.6
comm==0.2.3
debugpy==1.8.21
decorator==5.3.1
defusedxml==0.7.1
distro==1.9.0
exceptiongroup==1.3.1
executing==2.2.1
faiss-cpu==1.15.0
fastjsonschema==2.22.2
fqdn==1.5.1
frozenlist==1.8.0
greenlet==3.5.5
h11==0.16.0
httpcore==1.0.9
httpcore2==2.12.0
httpx==0.28.1
httpx-sse==0.4.3
httpx2==2.12.0
idna==3.19
ipykernel==7.3.0
ipython==8.39.0
isoduration==20.11.0
jedi==0.20.0
Jinja2==3.1.6
jiter==0.16.0
json5==0.15.0
jsonpatch==1.33
jsonpointer==3.1.1
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
jupyter-events==0.12.1
jupyter-lsp==2.3.1
jupyter_builder==1.2.2
jupyter_client==8.10.0
jupyter_core==5.9.1
jupyter_server==2.21.0
jupyter_server_

In [24]:
import os
import sys
from dotenv import load_dotenv

In [25]:
# Load environment variables
load_dotenv()

True

In [26]:
# Import LangChain components
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [5]:
# For DeepSeek API (https://platform.deepseek.com/)
# Sign up for free credits
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "")
DEEPSEEK_API_BASE = "https://api.deepseek.com/v1"

In [6]:
class RAGConfig:
    """Configuration for RAG system with free API support"""
    
    # Choose your API provider
    API_PROVIDER = "deepseek"  # Options: "deepseek", "groq", "gemini", "ollama"
    
    # Model settings
    if API_PROVIDER == "deepseek":
        MODEL_NAME = "deepseek-chat"  # DeepSeek's free model
        API_KEY = os.getenv("DEEPSEEK_API_KEY", "")
        BASE_URL = "https://api.deepseek.com/v1"
        EMBEDDING_MODEL = "text-embedding-3-small"  # or use local embeddings
    
    elif API_PROVIDER == "groq":
        MODEL_NAME = "mixtral-8x7b-32768"  # Free Groq model
        API_KEY = os.getenv("GROQ_API_KEY", "")
        BASE_URL = "https://api.groq.com/openai/v1"
        EMBEDDING_MODEL = "text-embedding-3-small"
    
    elif API_PROVIDER == "ollama":
        MODEL_NAME = "llama2"  # Local model
        API_KEY = "ollama"  # Not needed for local
        BASE_URL = "http://localhost:11434/v1"
        EMBEDDING_MODEL = "text-embedding-3-small"
    
    else:  # Default to OpenAI compatible
        MODEL_NAME = "gpt-3.5-turbo"
        API_KEY = os.getenv("OPENAI_API_KEY", "")
        BASE_URL = "https://api.openai.com/v1"
        EMBEDDING_MODEL = "text-embedding-3-small"
    
    # RAG settings
    CHUNK_SIZE = 500
    CHUNK_OVERLAP = 50
    RETRIEVAL_K = 5
    TEMPERATURE = 0.3
    MAX_TOKENS = 500

config = RAGConfig()

In [7]:
# Set environment variables for API access
if config.API_KEY:
    os.environ["OPENAI_API_KEY"] = config.API_KEY
    os.environ["OPENAI_API_BASE"] = config.BASE_URL

In [8]:
# Create embeddings (using OpenAI compatible API)
embeddings = OpenAIEmbeddings(
    model=config.EMBEDDING_MODEL,
    openai_api_key=config.API_KEY if config.API_KEY else "dummy",
    openai_api_base=config.BASE_URL,
)

In [9]:
# Create LLM instance (using the configured API)
llm = ChatOpenAI(
    model=config.MODEL_NAME,
    temperature=config.TEMPERATURE,
    max_tokens=config.MAX_TOKENS,
    openai_api_key=config.API_KEY if config.API_KEY else "dummy",
    openai_api_base=config.BASE_URL,
)

In [10]:
print(f"✅ Using API: {config.API_PROVIDER}")
print(f"✅ Model: {config.MODEL_NAME}")
print(f"✅ Base URL: {config.BASE_URL}")

✅ Using API: deepseek
✅ Model: deepseek-chat
✅ Base URL: https://api.deepseek.com/v1


In [11]:
# ## 6. Document Loading and Processing Functions

# %%
def load_documents(docs_path="./docs/"):
    """Load documents from directory"""
    loader = DirectoryLoader(
        docs_path,
        glob="**/*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    documents = loader.load()
    print(f"📄 Loaded {len(documents)} documents")
    return documents

def split_documents(documents):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP,
        separators=["\n\n", "\n", "。", "！", "？", "，", " ", ""],
        length_function=len
    )
    chunks = text_splitter.split_documents(documents)
    print(f"✂️ Split into {len(chunks)} chunks")
    return chunks


In [12]:
# Test document loading (create sample docs if none exist)
import os

# Create sample documents if they don't exist
if not os.path.exists("./docs"):
    os.makedirs("./docs")
    sample_docs = [
        "Artificial Intelligence (AI) is the simulation of human intelligence in machines.",
        "Machine Learning is a subset of AI that enables systems to learn from data.",
        "Deep Learning uses neural networks with multiple layers to process complex patterns.",
        "Natural Language Processing (NLP) helps computers understand human language.",
        "RAG (Retrieval-Augmented Generation) combines retrieval and generation for better answers."
    ]
    
    for i, text in enumerate(sample_docs):
        with open(f"./docs/sample_{i+1}.txt", "w", encoding="utf-8") as f:
            f.write(text)
    print("✅ Created sample documents")

In [13]:
# ## 7. Vector Store Creation

# %%
def create_vector_store(chunks):
    """Create or load FAISS vector store"""
    # Check if vector store exists
    if os.path.exists("./db/faiss_index"):
        try:
            vectorstore = FAISS.load_local(
                "./db/faiss_index",
                embeddings,
                allow_dangerous_deserialization=True
            )
            print("✅ Loaded existing vector store")
            return vectorstore
        except:
            print("⚠️ Could not load existing store, creating new one")
    
    # Create new vector store
    try:
        vectorstore = FAISS.from_documents(chunks, embeddings)
        # Save the vector store
        os.makedirs("./db", exist_ok=True)
        vectorstore.save_local("./db/faiss_index")
        print("✅ Created and saved new vector store")
        return vectorstore
    except Exception as e:
        print(f"⚠️ Error creating vector store: {e}")
        # Fallback: Use simple retriever without embeddings
        return None

In [14]:
# ## 8. RAG Chain Creation

# %%
def create_rag_chain(vectorstore):
    """Create RAG chain with error handling for free APIs"""
    
    if vectorstore is None:
        print("⚠️ No vector store available. Using simple LLM without retrieval.")
        # Simple chain without RAG
        prompt = ChatPromptTemplate.from_template(
            "Question: {question}\nAnswer concisely:"
        )
        chain = prompt | llm | StrOutputParser()
        return chain
    
    try:
        # Create retriever
        retriever = vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={
                "k": config.RETRIEVAL_K,
                "fetch_k": 20,
                "lambda_mult": 0.5
            }
        )
        
        # Prompt template
        template = """You are an expert assistant for question-answering tasks.
        Use the following pieces of retrieved context to answer the question.
        If you don't know the answer, just say "I don't have enough information to answer that."
        Keep your answer concise and cite the source when possible.
        
        Question: {question}
        
        Context: {context}
        
        Answer:"""
        
        prompt = ChatPromptTemplate.from_template(template)
        
        # Format documents
        def format_docs(docs):
            return "\n\n".join([
                f"Source: {doc.metadata.get('source', 'Unknown')}\n{doc.page_content}"
                for doc in docs
            ])
        
        # Build RAG chain
        rag_chain = (
            {
                "context": retriever | format_docs,
                "question": RunnablePassthrough()
            }
            | prompt
            | llm
            | StrOutputParser()
        )
        
        return rag_chain
        
    except Exception as e:
        print(f"⚠️ Error creating RAG chain: {e}")
        # Fallback to simple chain
        prompt = ChatPromptTemplate.from_template(
            "Question: {question}\nAnswer based on general knowledge:"
        )
        chain = prompt | llm | StrOutputParser()
        return chain


In [15]:
# ## 9. Initialize the RAG System

# %%
# Load and process documents
print("\n=== 📚 Setting up RAG System ===\n")


=== 📚 Setting up RAG System ===



In [27]:
# ## 10. Question Answering Function

# %%
def ask_question(question, rag_chain=rag_chain if 'rag_chain' in locals() else None):
    """Ask a question and get answer"""
    if rag_chain is None:
        return "⚠️ RAG system not properly initialized. Please check your setup."
    
    try:
        answer = rag_chain.invoke(question)
        return answer
    except Exception as e:
        error_msg = str(e)
        if "authentication" in error_msg.lower() or "api key" in error_msg.lower():
            return f"⚠️ API authentication error: {error_msg}\n\nPlease set your API key correctly or try a different API provider."
        elif "rate limit" in error_msg.lower():
            return "⚠️ Rate limit exceeded. Please wait a moment and try again."
        else:
            return f"⚠️ Error: {error_msg}"

In [28]:
# ## 11. Interactive Q&A

# %%
# Interactive Q&A
print("\n" + "="*50)
print("🎯 RAG Q&A System - Interactive Mode")
print("="*50)
print("\nType your questions below. Type 'exit' to quit.\n")

# Check if system is initialized
if 'rag_chain' not in locals() or rag_chain is None:
    print("⚠️ System not initialized. Please check your setup.")
else:
    while True:
        try:
            # Get user input
            question = input("\n❓ Your question: ")
            
            # Check for exit
            if question.lower() in ["exit", "quit", "q"]:
                print("\n👋 Goodbye!")
                break
            
            # Skip empty questions
            if not question.strip():
                continue
            
            # Get answer
            print("\n💭 Thinking...")
            answer = ask_question(question)
            print(f"\n✅ Answer:\n{answer}\n")
            print("-"*50)
            
        except KeyboardInterrupt:
            print("\n\n👋 Goodbye!")
            break
        except Exception as e:
            print(f"\n⚠️ Unexpected error: {e}\n")


🎯 RAG Q&A System - Interactive Mode

Type your questions below. Type 'exit' to quit.

⚠️ System not initialized. Please check your setup.


In [18]:
# ## 12. Batch Question Answering

# %%
def answer_multiple_questions(questions_list):
    """Answer multiple questions at once"""
    if 'rag_chain' not in locals() or rag_chain is None:
        return "⚠️ RAG system not initialized"
    
    results = {}
    for q in questions_list:
        results[q] = ask_question(q)
    return results


In [19]:
# ## 13. Free API Setup Guide

# %% [markdown]
# ### Free API Options:
# 
# **1. DeepSeek API (Recommended)**
# - Sign up at: https://platform.deepseek.com/
# - Get free credits to start
# - Set environment variable: `DEEPSEEK_API_KEY`
# 
# **2. Groq API**
# - Sign up at: https://console.groq.com/
# - Free tier with good models
# - Set environment variable: `GROQ_API_KEY`
# 
# **3. Ollama (Local)**
# - Install Ollama: https://ollama.ai/
# - Pull a model: `ollama pull llama2`
# - Run: `ollama serve`
# 
# **4. Gemini API**
# - Google AI Studio: https://makersuite.google.com/
# - Free tier available
# - Set environment variable: `GEMINI_API_KEY`
# 
# ### Setting Environment Variables:
# 
# Create a `.env` file in your project root:
# ```
# DEEPSEEK_API_KEY=your_api_key_here
# # or
# GROQ_API_KEY=your_api_key_here
# ```

In [20]:
# ## 14. System Health Check

# %%
def check_system_health():
    """Check if all components are working"""
    print("\n🔍 System Health Check")
    print("="*40)
    
    checks = []
    
    # Check API key
    if config.API_KEY:
        print("✅ API Key: Configured")
        checks.append(True)
    else:
        print("⚠️ API Key: Missing (using fallback)")
        checks.append(False)
    
    # Check documents
    if os.path.exists("./docs"):
        doc_count = len([f for f in os.listdir("./docs") if f.endswith(".txt")])
        print(f"✅ Documents: {doc_count} files found")
        checks.append(True)
    else:
        print("⚠️ Documents: No ./docs folder found")
        checks.append(False)
    
    # Check vector store
    if os.path.exists("./db/faiss_index"):
        print("✅ Vector Store: Available")
        checks.append(True)
    else:
        print("⚠️ Vector Store: Not created yet")
        checks.append(False)
    
    # Check LLM connectivity
    try:
        test_response = llm.invoke("Say 'test' if you can hear me")
        print("✅ LLM: Responding")
        checks.append(True)
    except Exception as e:
        print(f"⚠️ LLM: Connection error - {str(e)[:50]}...")
        checks.append(False)
    
    # Summary
    print("\n" + "="*40)
    health_score = sum(checks) / len(checks) * 100
    print(f"Health Score: {health_score:.0f}%")
    if health_score < 50:
        print("⚠️ System needs attention. Check API configuration.")
    else:
        print("✅ System is ready for use!")
    
    return health_score
# Run health check
health_score = check_system_health()


🔍 System Health Check
✅ API Key: Configured
✅ Documents: 1 files found
⚠️ Vector Store: Not created yet
⚠️ LLM: Connection error - Error code: 402 - {'error': {'message': 'Insuffici...

Health Score: 50%
✅ System is ready for use!


In [21]:
# ## 15. Additional Utilities

# %%
def create_sample_documents():
    """Create sample documents for testing"""
    os.makedirs("./docs", exist_ok=True)
    
    samples = {
        "ai_intro.txt": """
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        The goals of AI include learning, reasoning, and perception.
        AI is being used across industries including healthcare, finance, and transportation.
        """,
        
        "machine_learning.txt": """
        Machine Learning is a subset of AI that enables systems to learn from data.
        It uses algorithms to find patterns in data without being explicitly programmed.
        Common types include supervised learning, unsupervised learning, and reinforcement learning.
        """,
        
        "deep_learning.txt": """
        Deep Learning is a subset of machine learning that uses neural networks.
        These networks have multiple layers that can learn hierarchical representations.
        Deep learning has been particularly successful in computer vision and NLP tasks.
        """,
        
        "nlp.txt": """
        Natural Language Processing (NLP) helps computers understand human language.
        It combines linguistics, computer science, and AI to process text and speech.
        Applications include translation, sentiment analysis, and chatbots.
        """,
        
        "rag.txt": """
        RAG (Retrieval-Augmented Generation) combines information retrieval with language generation.
        It retrieves relevant documents and uses them to generate informed responses.
        RAG reduces hallucinations by grounding answers in retrieved content.
        """
    }
    
    for filename, content in samples.items():
        with open(f"./docs/{filename}", "w", encoding="utf-8") as f:
            f.write(content)
    
    print(f"✅ Created {len(samples)} sample documents in ./docs/")

# Uncomment to create sample documents
create_sample_documents()

✅ Created 5 sample documents in ./docs/


In [22]:
# ## 16. Quick Test

# %%
# Quick test if system is ready
if 'rag_chain' in locals() and rag_chain is not None:
    test_question = "What is RAG?"
    print("\n🧪 Quick Test:")
    print(f"Q: {test_question}")
    try:
        answer = ask_question(test_question)
        print(f"A: {answer}")
    except Exception as e:
        print(f"⚠️ Test failed: {e}")
else:
    print("⚠️ System not ready. Run the initialization cells first.")

⚠️ System not ready. Run the initialization cells first.
